# Introduction to chemoinformatics with RDKit

## Configuration

In [ ]:
# Magic commands
%load_ext autoreload
%autoreload 2

In [ ]:
# Imports
# 1. Standard library imports
from pathlib import Path
import sys
sys.path.append('../my_modules') # to tell where to find local modules

# 2. Third-party library imports
import pandas as pd
import rdkit
from rdkit import Chem
from rdkit.Chem import (
    AllChem,
    Descriptors,
    Draw,
    PandasTools
)
PandasTools.RenderImagesInAllDataFrames(images=True) # to molecules as images in DataFrames
from rdkit.Chem.Draw import IPythonConsole # needed to show molecules
# from rdkit.Chem.Draw.MolDrawing import MolDrawing, DrawingOptions # only needed if modifying defaults

# 3. Local application imports
import kernel_infos

In [ ]:
# Information about the kernel
kernel_infos.show_kernel_info()

In [ ]:
# Global variables
HERE = Path().resolve()
print(f"HERE: {HERE}")
ROOT = HERE.parent
print(f"ROOT: {ROOT}")
DATA = ROOT / 'data'
print(f"DATA: {DATA}")

## Objective
Manipulate molecules with RDKit: open source toolkit for chemoinformatics
- reading, writing, drawing
- compute molecular descriptors

## Documentation
- [RDKit documentation](http://www.rdkit.org/docs/index.html)
- [Python API Reference](http://www.rdkit.org/docs/api/index.html)
- [RDKit cookbook](https://www.rdkit.org/docs/Cookbook.html)

In [ ]:
# RDKit version
print(f"RDKit version: {rdkit.__version__}")

## Molecular representation

### Structural identifiers

From [Chemspider](https://www.chemspider.com/Chemical-Structure.2157.html) get smiles of aspirin.

In [ ]:
# SMILES
aspirin_smi = 'CC(=O)Oc1ccccc1C(=O)O'

Create RDKit molecule object and visualise it

In [ ]:
# Molecule
aspirin_mol = Chem.MolFromSmiles(aspirin_smi)

In [ ]:
# Visualise molecule
aspirin_mol

Compute and print inchi

In [ ]:
aspirin_inchi = Chem.MolToInchi(aspirin_mol)
print(f"Inchi: {aspirin_inchi}")

Compute and print inchikey

In [ ]:
aspirin_inchikey = Chem.MolToInchiKey(aspirin_mol)
print(f"InchiKey: {aspirin_inchikey}")

### Mol file

From [Chemspider](https://www.chemspider.com/Chemical-Structure.2157.html) you can download download .mol (data/2157.mol).

- View 2157.mol in a text editor
- What is the dimensionality of the molecule ?

Generate and print aspirin molblock and compare to 2157.mol

In [ ]:
aspirin_molblock = Chem.MolToMolBlock(aspirin_mol)
print(f"Aspirin molblock: {aspirin_molblock}")

Save aspirin molecule in a mol file

In [ ]:
# Define a mol file path
mol_file_path = DATA / "aspirin2D.mol"

# Save file
Chem.MolToMolFile(
    aspirin_mol,
    mol_file_path
)

### 3D

Generate 3D coordinates

In [ ]:
# Add explicit hydrogens and visualise molecule
aspirin_3D_mol = Chem.AddHs(aspirin_mol)
aspirin_3D_mol

In [ ]:
# Generate an initial 3D conformation and optimize it with MMFF
AllChem.EmbedMolecule(aspirin_3D_mol)
AllChem.MMFFOptimizeMolecule(aspirin_3D_mol)

Please note that the previous functions do not return molecules. They modify them in situ.

In [ ]:
# Visualise the resulting conformation
aspirin_3D_mol

Save the 3D conformation in a mol file and compare with aspirin2D.mol

In [ ]:
# Define a mol file path
mol_file_path = DATA / "aspirin3D.mol"

# Save file
Chem.MolToMolFile(
    aspirin_3D_mol,
    mol_file_path
)

## Molecular data

Inspect molecule object

In [ ]:
# molecule object
dir(rdkit.Chem.rdchem.Mol)

Compute and print the number of atoms in the aspirin molecule

In [ ]:
nb_atoms = aspirin_mol.GetNumAtoms()
print(f"Number of atoms: {nb_atoms}")

In [ ]:
help(aspirin_mol.GetNumAtoms)

Compute and print the number of bonds in the aspirin molecule

In [ ]:
nb_bonds = aspirin_mol.GetNumBonds()
print(f"Number of bonds: {nb_bonds}")

In [ ]:
aspirin_mol

List the atoms symbols

In [ ]:
# Atom object
dir(rdkit.Chem.rdchem.Atom)

In [ ]:
for index, atom in enumerate(aspirin_mol.GetAtoms()):
    print(f"Atom {index}: {atom.GetSymbol()} ")

## Descriptors
### Chem.Descriptors.rdMolDescriptors
This module provides basic molecular descriptors, typically related to 2D structure or simple topological properties.

Example Descriptors:

- CalcMolWt(mol): Molecular weight.
- CalcNumRotatableBonds(mol): Number of rotatable bonds.
- CalcNumHBA(mol): Number of hydrogen bond acceptors.
- CalcNumHBD(mol): Number of hydrogen bond donors.
- CalcTPSA(mol): Topological polar surface area (TPSA).

Advantages:

- Fast and optimized for simple calculations.
- Ideal for basic structural analysis.

In [ ]:
dir(Descriptors.rdMolDescriptors)

Compute and print aspirin number of hydrogen bond acceptors

In [ ]:
HBA = Descriptors.rdMolDescriptors.CalcNumHBA(aspirin_mol)
print(f"Aspirin HBA: {HBA}")

Compute and print aspirin number of hydrogen bond donors

In [ ]:
HBD = Descriptors.rdMolDescriptors.CalcNumHBD(aspirin_mol)
print(f"Aspirin HBD: {HBD}")

### Chem.Descriptors

This module offers a broader collection of descriptors, including physicochemical properties, 2D/3D descriptors, and QSAR (Quantitative Structure-Activity Relationship) descriptors.

Example Descriptors:

- MolLogP(mol): LogP (octanol/water partition coefficient).
- NumHAcceptors(mol): Number of hydrogen bond acceptors (similar to rdMolDescriptors but sometimes with different calculation nuances).
- NumHDonors(mol): Number of hydrogen bond donors.
- NumRotatableBonds(mol): Number of rotatable bonds (often more detailed).

Advantages:

- More comprehensive, with advanced descriptors for molecular modeling.
- Includes descriptors used in predictive models (e.g., LogP).

In [ ]:
dir(Descriptors)

Compute and print aspirin LogP

Definition of LogP:
LogP is a measure of a molecule's lipophilicity, which is its tendency to dissolve in organic solvents (such as octanol) rather than in water.
It is defined as the base-10 logarithm of the ratio of a molecule's concentrations in an octanol/water mixture at equilibrium:
$\text{LogP} = \log_{10}\left(\frac{[\text{solute}]_{\text{octanol}}}{[\text{solute}]_{\text{water}}}\right)$

Molecules with an optimal LogP (typically between 1 and 3) have a better chance of crossing cell membranes and being absorbed by the body.
- A LogP that is too high (molecule too lipophilic) can lead to poor water solubility and accumulation in fatty tissues.
- A LogP that is too low (molecule too hydrophilic) can limit cellular penetration.


In [ ]:
# Octanol / Water partition coefficient
LogP = Descriptors.MolLogP(aspirin_mol)
print(f"Aspirin MolLogP: {LogP}")

Compute and print aspirin heavy atom count: HAC

In [ ]:
# Number of heavy atom (all but not H atoms)
HAC = Descriptors.HeavyAtomCount(aspirin_mol)
print(f"Aspirin number of heavy atoms: {HAC}")

### Summary

|Criteria|rdMolDescriptors|Descriptors|
|:--|:--|:--|
|Descriptor Type|Basic, structural|Advanced, physicochemical, QSAR|
|Examples|Molecular weight, rotatable bonds|LogP, 3D descriptors|
|Optimization|Fast, lightweight|More comprehensive, sometimes slower|
|Use Case|Simple structural analysis|Modeling, QSAR, predictive analysis|

## Stereochemistry

![Isomerism](./images/isomerism.png)

Thalidomide is a drug that was used in the 1950s and 1960s as an anti-nausea medication for pregnant women and as a sedative.

It was discovered to cause severe birth defects. These teratogenic effects were initially concealed or denied, particularly by the manufacturer Grünenthal GmbH (de). Subsequently, they became the subject of a health scandal that led to the drug being withdrawn from the global market in 1961. Today, thalidomide is used in a highly controlled manner for its immunomodulatory and antitumour properties.

In [ ]:
# Thalidomide
thalidomide_smi = 'O=C1N(C(CC2)C(NC2=O)=O)C(C3=C1C=CC=C3)=O'

# Sedative form
R_thalidomide_smi = 'O=C1N([C@H](CC2)C(NC2=O)=O)C(C3=C1C=CC=C3)=O'

# Teratogenic form
S_thalidomide_smi = 'O=C1N([C@@H](CC2)C(NC2=O)=O)C(C3=C1C=CC=C3)=O'

In [ ]:
thalidomide_mol = Chem.MolFromSmiles(thalidomide_smi)
thalidomide_mol

In [ ]:
R_thalidomide_mol = Chem.MolFromSmiles(R_thalidomide_smi)
R_thalidomide_mol

In [ ]:
S_thalidomide_mol = Chem.MolFromSmiles(S_thalidomide_smi)
S_thalidomide_mol

## Work with several molecules

### Dictionary to Dataframe

In [ ]:
drugs_dict = {
    'name':['Paracetamol', 'Ibuprofen', 'Amoxicillin', 'Atorvastatin', 'Omeprazole'],
    'smiles':[
        'CC(=O)Nc1ccc(O)cc1',
        'CC(C)Cc1ccc(C(C)C(=O)O)cc1',
        'CC1(C)S[C@@H]2[C@H](NC(=O)[C@H](N)c3ccc(O)cc3)C(=O)N2[C@H]1C(=O)O',
        'CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F)cc2)n1CC[C@@H](O)C[C@@H](O)CC(=O)O',
        'COc1ccc2[nH]c(S(=O)Cc3ncc(C)c(OC)c3C)nc2c1'
    ],
    'indication':[
        'Pain reliever and fever reducer',
        'Anti-inflammatory',
        'antibiotic',
        'Cholesterol-lowering',
        'treatment of ulcers and gastroesophageal reflux disease'
    ]    
}

In [ ]:
drugs_df = pd.DataFrame.from_dict(drugs_dict)

In [ ]:
# Check
print(f"drugs_df shape: {drugs_df.shape}")
drugs_df

### DataFrame to CSV file

In [ ]:
# File path
csv_file_path = DATA / 'drugs.csv'

In [ ]:
# Export Dataframe to csv file
drugs_df.to_csv(
    csv_file_path,
    sep=',',
    header=True,
    index=False
    )

### Molecular dataframe

In [ ]:
# Add a colum (ROMol) containing molecule objects using the function MolFromSmiles 
drugs_df['ROMol'] = drugs_df['smiles'].map(Chem.MolFromSmiles)

In [ ]:
drugs_df

In [ ]:
# Add a colum (ROMol2) containing molecule objects using PandasTools
PandasTools.AddMoleculeColumnToFrame(
    drugs_df,
    smilesCol='smiles',
    molCol='ROMol2',
    includeFingerprints=False,
)

In [ ]:
# Check
print(f"drugs_df shape: {drugs_df.shape}")
drugs_df

In [ ]:
# Delete the last added column
drugs_df.drop(
    'ROMol2',
    axis='columns',
    inplace=True,
)

In [ ]:
# Check
print(f"drugs_df shape: {drugs_df.shape}")
drugs_df

### SDF format

In [ ]:
# sdf_file_path (example from Ambinter)
sdf_file_path = DATA/ "ambinter_example.sdf"

#### SDMolSupplier

In [ ]:
# Define a file supplier
suppl = Chem.SDMolSupplier(str(sdf_file_path))

molecules_list = list()
for mol in suppl:
    if mol is not None:
        molecules_list.append(mol)
        print(f"Name: {mol.GetProp('_Name')}") if mol.HasProp('_Name') else print("No name")

        props = mol.GetPropsAsDict()
        for key, value in props.items():
            print(f"{key}: {value}")

In [ ]:
# Display molecules_list
for ind, mol in enumerate(molecules_list):
    print(f"molecule {ind}: {mol}")

In [ ]:
# Display the first molecule
molecules_list[0]

In [ ]:
# Display the _Name property of the first molecule
molecules_list[0].GetProp('_Name')

#### PandasTools

In [ ]:
# Load sdf file in a DataFrame
ambinter_df = PandasTools.LoadSDF(
    str(sdf_file_path)
)

In [ ]:
# Check the DataFrame
print(f"ambinter_df shape: {ambinter_df.shape}")
ambinter_df

In [ ]:
# First molecule in ambinter_df
ambinter_df.loc[0, 'ROMol']

In [ ]:
# Load sdf file in a DataFrame and add a smiles column
ambinter_df = PandasTools.LoadSDF(
    str(sdf_file_path),
    smilesName='smiles'   
)

In [ ]:
# Check the DataFrame
print(f"ambinter_df shape: {ambinter_df.shape}")
ambinter_df

In [ ]:
# Load sdf file in a DataFrame with embedProps=True
ambinter_df = PandasTools.LoadSDF(
    str(sdf_file_path),
    embedProps=True
)

In [ ]:
# Check the DataFrame
print(f"ambinter_df shape: {ambinter_df.shape}")
ambinter_df

In [ ]:
# First molecule in ambinter_df
ambinter_df.loc[0, 'ROMol']

### Visualisation / images

In [ ]:
# Save molecules from drugs_df in a grid image
grid_image = Draw.MolsToGridImage(
    mols = drugs_df['ROMol'].to_list(),
    legends=drugs_df['name'].to_list(),
    molsPerRow=3,
    subImgSize=(200,200)
)

In [ ]:
# Show the image
grid_image

In [ ]:
# Save the image in a png file
png_file_path = DATA / 'drugs_grid.png'

with open(png_file_path, 'wb') as f:
    f.write(grid_image.data)